# Model Inference TimesFM (Walmart Sales Forecasting)

In [1]:
!pip install "timesfm[torch,xreg]" mlflow dagshub -q

import numpy as np
import pandas as pd
import mlflow
import dagshub
from mlflow.tracking import MlflowClient

dagshub.init(repo_owner='ejoba22', repo_name='walmart-sales-forecasting', mlflow=True)
print("Tracking URI:", mlflow.get_tracking_uri())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 95.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 89.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=7bbfacc7-12d7-4ec4-9894-d56baa14f045&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=da649b8e03d757aa61eee27e73bc88754647df025ff32d27c4feecb372a7ed8e




Accessing as ejoba22

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

Tracking URI: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow


## Model Registry

In [2]:
MODEL_NAME = "walmart_timesfm_pipeline"  

client = MlflowClient()

versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max(int(v.version) for v in versions)
model_uri = f"models:/{MODEL_NAME}/{latest_version}"

print(f"Loading {model_uri}")
model = mlflow.pyfunc.load_model(model_uri)

Loading models:/walmart_timesfm_pipeline/2


In [7]:
RAW_BASE = '/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/'

raw_train = pd.read_csv(RAW_BASE + 'train.csv.zip')
raw_train['Date'] = pd.to_datetime(raw_train['Date'])

print(raw_train.shape)
raw_train.head()

(421570, 5)


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


## Inference

In [8]:
predictions = model.predict(raw_train)

print(predictions.shape)
predictions.head()

2026/07/12 16:25:43 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['IsHoliday']`. These inputs will be ignored.


(130338, 4)


,Store,Dept,Date,Weekly_Sales
0,1,1,2012-11-02,28389.703125
1,1,1,2012-11-09,23287.589844
2,1,1,2012-11-16,18871.216797
3,1,1,2012-11-23,19423.316406
4,1,1,2012-11-30,24692.972656


## Kaggle Submission 

In [9]:
sample_submission = pd.read_csv(RAW_BASE + 'sampleSubmission.csv.zip')

predictions['Id'] = (
    predictions['Store'].astype(str) + '_' +
    predictions['Dept'].astype(str) + '_' +
    predictions['Date'].dt.strftime('%Y-%m-%d')
)

pred_lookup = predictions.set_index('Id')['Weekly_Sales']

submission = sample_submission.copy()
submission['Weekly_Sales'] = submission['Id'].map(pred_lookup)

n_missing = submission['Weekly_Sales'].isna().sum()
if n_missing:
    print(f"{n_missing} Id ვერ დაიფარა pipeline-ის მიერ — ივსება 0-ით fallback-ად.")
    submission['Weekly_Sales'] = submission['Weekly_Sales'].fillna(0)

submission.to_csv('submission.csv', index=False)
submission.head()

,Id,Weekly_Sales
0,1_1_2012-11-02,28389.703125
1,1_1_2012-11-09,23287.589844
2,1_1_2012-11-16,18871.216797
3,1_1_2012-11-23,19423.316406
4,1_1_2012-11-30,24692.972656
